# Phase 2: Data Cleaning & Preprocessing
## E-Commerce Sales & Customer Analytics Dashboard

This notebook implements production-quality cleaning steps on the Olist dataset:
- Missing value treatment
- Duplicate record removal
- DateTime parsing & formatting
- English category translations
- Outlier detection & winsorization (capping)
- Data integrity & schema validation

Cleaned tables will be saved in `data/cleaned/` to serve as our analytics base.

---

In [1]:
import pandas as pd
import numpy as np
import os

RAW_DATA_DIR = '../data/raw/'
CLEANED_DATA_DIR = '../data/cleaned/'
os.makedirs(CLEANED_DATA_DIR, exist_ok=True)

# Load raw tables
customers = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_customers_dataset.csv'))
orders = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_orders_dataset.csv'))
order_items = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_order_items_dataset.csv'))
order_payments = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_order_payments_dataset.csv'))
order_reviews = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_order_reviews_dataset.csv'))
products = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_products_dataset.csv'))
sellers = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_sellers_dataset.csv'))
geolocation = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_geolocation_dataset.csv'))
category_translation = pd.read_csv(os.path.join(RAW_DATA_DIR, 'product_category_name_translation.csv'))

print('Successfully loaded all datasets')

Successfully loaded all datasets


### 1. DateTime Formatting
Convert string timestamps to datetime format for calculations. Orders table contains multiple timestamps.

In [2]:
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')
    
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'], errors='coerce')
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'], errors='coerce')

print(orders[datetime_cols].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


### 2. Missing Value Imputation
Handling nulls in reviews, products, and order timestamps.

In [3]:
# Reviews: Impute empty comment text with empty string
order_reviews['review_comment_title'] = order_reviews['review_comment_title'].fillna('')
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].fillna('')

# Products: Impute missing product metrics with median, categories with 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')
product_numeric = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 
                   'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in product_numeric:
    products[col] = products[col].fillna(products[col].median())

print('Null values treated in reviews and products.')

Null values treated in reviews and products.


### 3. Duplicate Resolution & Geolocation Aggregation
The `geolocation` dataset contains massive duplicate rows for the same `zip_code_prefix`. We will aggregate the latitude, longitude and take the first city/state to create a clean dimensions table.

In [4]:
print(f'Original geolocation size: {geolocation.shape[0]} rows')

geolocation_cleaned = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

print(f'Cleaned geolocation size (unique zip codes): {geolocation_cleaned.shape[0]} rows')

# Drop duplicates in all other tables if any exist
customers = customers.drop_duplicates()
orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()
order_payments = order_payments.drop_duplicates()
order_reviews = order_reviews.drop_duplicates()
products = products.drop_duplicates()
sellers = sellers.drop_duplicates()

Original geolocation size: 1000163 rows
Cleaned geolocation size (unique zip codes): 19015 rows


### 4. Category Name Standardization & Translation
Map Portuguese product categories to English using `category_translation`. If a mapping is missing, we use the original name.

In [5]:
translation_dict = dict(zip(category_translation['product_category_name'], category_translation['product_category_name_english']))

products['product_category_name_english'] = products['product_category_name'].map(translation_dict)
# If no translation found, fill with original name
products['product_category_name_english'] = products['product_category_name_english'].fillna(products['product_category_name'])
# Standardize naming format (title case, replace underscores)
products['product_category_name_english'] = products['product_category_name_english'].str.replace('_', ' ').str.title()

print('Categories translated and formatted. Top 5 categories:')
print(products['product_category_name_english'].value_counts().head(5))

Categories translated and formatted. Top 5 categories:
product_category_name_english
Bed Bath Table     3029
Sports Leisure     2867
Furniture Decor    2657
Health Beauty      2444
Housewares         2335
Name: count, dtype: int64


### 5. Outlier Treatment (Price & Freight)
Identify outliers in prices and freight value. Instead of deleting transactional records, we will caps price and freight values at the 99th percentile (Winsorization) to prevent extreme outliers from skewing average metrics while keeping full revenue histories.

In [6]:
price_cap = order_items['price'].quantile(0.99)
freight_cap = order_items['freight_value'].quantile(0.99)

print(f'99th percentile for price: {price_cap:.2f} BRL')
print(f'99th percentile for freight: {freight_cap:.2f} BRL')

order_items['price_capped'] = np.where(order_items['price'] > price_cap, price_cap, order_items['price'])
order_items['freight_capped'] = np.where(order_items['freight_value'] > freight_cap, freight_cap, order_items['freight_value'])

print('Outlier capping complete.')

99th percentile for price: 890.00 BRL
99th percentile for freight: 84.52 BRL
Outlier capping complete.


### 6. Export Cleaned Datasets
Write files out for database importing and feature engineering.

In [7]:
customers.to_csv(os.path.join(CLEANED_DATA_DIR, 'customers_cleaned.csv'), index=False)
orders.to_csv(os.path.join(CLEANED_DATA_DIR, 'orders_cleaned.csv'), index=False)
order_items.to_csv(os.path.join(CLEANED_DATA_DIR, 'order_items_cleaned.csv'), index=False)
order_payments.to_csv(os.path.join(CLEANED_DATA_DIR, 'order_payments_cleaned.csv'), index=False)
order_reviews.to_csv(os.path.join(CLEANED_DATA_DIR, 'order_reviews_cleaned.csv'), index=False)
products.to_csv(os.path.join(CLEANED_DATA_DIR, 'products_cleaned.csv'), index=False)
sellers.to_csv(os.path.join(CLEANED_DATA_DIR, 'sellers_cleaned.csv'), index=False)
geolocation_cleaned.to_csv(os.path.join(CLEANED_DATA_DIR, 'geolocation_cleaned.csv'), index=False)

print('All cleaned CSVs successfully exported to data/cleaned/')

All cleaned CSVs successfully exported to data/cleaned/
